In [6]:
import os
import cv2
import shutil
import albumentations as A
import random
import numpy as np
from tqdm import tqdm

# Augmentation Transformations (With Fixes)
augmentations = A.Compose([
    # **Structural Augmentations** (Only Vertical Flip because YOLOv8 already has horizontal flip)
    A.VerticalFlip(p=0.3),

    # **Mild Color & Lighting Adjustments**
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=10, p=0.4),  # Small color shifts

    # **Noise & Blur (Improves Robustness)**
    A.GaussNoise(var_limit=(10, 20), p=0.3),
    A.MotionBlur(blur_limit=3, p=0.2),  # Simulates motion blur

    # **Weather Augmentations (Softer Fog & Rain)**
    # A.RandomFog(fog_coef_lower=0.2, fog_coef_upper=0.4, alpha_coef=0.06, p=0.15),  # Lighter fog effect
    # A.RandomRain(
    #     rain_type=random.choice(["drizzle", "heavy"]),
    #     drop_length=15, drop_width=1, blur_value=2, brightness_coefficient=0.8, p=0.15
    # ),
], bbox_params=A.BboxParams(format='yolo', label_fields=['category_ids'], min_visibility=0.3))  # Increase min_visibility to 0.3

# Function to read YOLO bounding boxes
def read_yolo_bboxes(annotation_path, image_shape):
    h, w, _ = image_shape
    bboxes = []
    category_ids = []
    if not os.path.exists(annotation_path):
        return bboxes, category_ids

    with open(annotation_path, "r") as file:
        for line in file.readlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])
            bboxes.append([x_center, y_center, width, height])
            category_ids.append(class_id)

    return bboxes, category_ids


# Function to save YOLO bounding boxes
def save_yolo_bboxes(annotation_path, bboxes, category_ids):
    with open(annotation_path, "w") as file:
        for bbox, class_id in zip(bboxes, category_ids):
            file.write(f"{class_id} {' '.join(map(str, bbox))}\n")


# Function to apply augmentations while adjusting bounding boxes
def apply_augmentations(input_image_dir, input_annot_dir, output_image_dir, output_annot_dir, augmentation_ratio=1.0):
    os.makedirs(output_image_dir, exist_ok=True)
    os.makedirs(output_annot_dir, exist_ok=True)

    images = [f for f in os.listdir(input_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    num_images = len(images)

    if num_images == 0:
        print("No images found for augmentation.")
        return

    num_augmented = int(num_images * augmentation_ratio)
    selected_images = random.sample(images, num_augmented)

    for filename in tqdm(selected_images, desc=f"Augmenting {num_augmented} images from {input_image_dir}"):
        image_path = os.path.join(input_image_dir, filename)
        base_filename = os.path.splitext(filename)[0]
        annotation_path = os.path.join(input_annot_dir, base_filename + ".txt")  # YOLO format

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        # Read bounding boxes
        bboxes, category_ids = read_yolo_bboxes(annotation_path, image.shape)

        # Apply augmentation
        augmented = augmentations(image=image, bboxes=bboxes, category_ids=category_ids)
        transformed_image = augmented['image']
        transformed_bboxes = augmented['bboxes']

        # If no change happened, apply another random augmentation
        while np.array_equal(transformed_image, image):
            print(f"🔄 Reapplying augmentation for {filename} (No changes detected)")
            augmented = augmentations(image=image, bboxes=bboxes, category_ids=category_ids)
            transformed_image = augmented['image']
            transformed_bboxes = augmented['bboxes']

        # Convert back to BGR for saving
        transformed_image = cv2.cvtColor(transformed_image, cv2.COLOR_RGB2BGR)

        # Create new filenames
        new_image_filename = f"aug_{filename}"
        new_annotation_filename = f"aug_{base_filename}.txt"

        # Save the augmented image
        output_image_path = os.path.join(output_image_dir, new_image_filename)
        cv2.imwrite(output_image_path, transformed_image)

        # Save updated bounding boxes
        if transformed_bboxes:
            output_annot_path = os.path.join(output_annot_dir, new_annotation_filename)
            save_yolo_bboxes(output_annot_path, transformed_bboxes, category_ids)
        else:
            print(f"⚠ Warning: No valid bounding boxes after augmentation for {filename}")


# Function to copy original data
def copy_original_data(input_image_dir, input_annot_dir, output_image_dir, output_annot_dir):
    os.makedirs(output_image_dir, exist_ok=True)
    os.makedirs(output_annot_dir, exist_ok=True)

    for filename in tqdm(os.listdir(input_image_dir), desc=f"Copying original images from {input_image_dir}"):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            shutil.copy(os.path.join(input_image_dir, filename), os.path.join(output_image_dir, filename))

    for filename in tqdm(os.listdir(input_annot_dir), desc=f"Copying original annotations from {input_annot_dir}"):
        if filename.lower().endswith(('.xml', '.txt')):
            shutil.copy(os.path.join(input_annot_dir, filename), os.path.join(output_annot_dir, filename))

# Define directories
train_image_dir = 'train_album_intact/train/images'
train_annot_dir = 'train_album_intact/train/labels'

train_output_image_dir = 'new_datasets_album100/train/images'
train_output_annot_dir = 'new_datasets_album100/train/labels'

# Step 2: Apply augmentations (50% of the dataset)
apply_augmentations(train_image_dir, train_annot_dir, train_output_image_dir, train_output_annot_dir, augmentation_ratio=0.3)

/var/folders/tq/x2bwjy7n23g_0m5lry7hq38m0000gn/T/ipykernel_4930/3285399782.py:19: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 20), p=0.3),


Augmenting 484 images from train_album_intact/train/images:   4%|▎         | 18/484 [00:00<00:07, 58.37it/s]

🔄 Reapplying augmentation for Japan_004604.jpg (No changes detected)
🔄 Reapplying augmentation for Japan_010779.jpg (No changes detected)
⚠ Warning: No valid bounding boxes after augmentation for Japan_010045.jpg


Augmenting 484 images from train_album_intact/train/images:   9%|▉         | 45/484 [00:00<00:06, 72.52it/s]

⚠ Warning: No valid bounding boxes after augmentation for Japan_003465.jpg
⚠ Warning: No valid bounding boxes after augmentation for Japan_003345.jpg
⚠ Warning: No valid bounding boxes after augmentation for Japan_004137.jpg
🔄 Reapplying augmentation for Japan_001637.jpg (No changes detected)


...

🔄 Reapplying augmentation for Japan_007844.jpg (No changes detected)
🔄 Reapplying augmentation for Japan_012635.jpg (No changes detected)
⚠ Warning: No valid bounding boxes after augmentation for Japan_001408.jpg


Augmenting 484 images from train_album_intact/train/images:  96%|█████████▌| 463/484 [00:05<00:00, 86.80it/s]

🔄 Reapplying augmentation for Japan_001109.jpg (No changes detected)
⚠ Warning: No valid bounding boxes after augmentation for Japan_006231.jpg
🔄 Reapplying augmentation for 463.jpg (No changes detected)
🔄 Reapplying augmentation for 463.jpg (No changes detected)
🔄 Reapplying augmentation for Japan_011417.jpg (No changes detected)
⚠ Warning: No valid bounding boxes after augmentation for Japan_012529.jpg
🔄 Reapplying augmentation for 240.jpg (No changes detected)
🔄 Reapplying augmentation for Japan_008614.jpg (No changes detected)


Augmenting 484 images from train_album_intact/train/images: 100%|██████████| 484/484 [00:05<00:00, 83.82it/s]

🔄 Reapplying augmentation for Japan_005780.jpg (No changes detected)
🔄 Reapplying augmentation for Japan_005780.jpg (No changes detected)
🔄 Reapplying augmentation for aug_Japan_007844.jpg (No changes detected)
⚠ Warning: No valid bounding boxes after augmentation for Japan_004286.jpg
⚠ Warning: No valid bounding boxes after augmentation for aug_Japan_000803.jpg
⚠ Warning: No valid bounding boxes after augmentation for Japan_000803.jpg
🔄 Reapplying augmentation for 489.jpg (No changes detected)
⚠ Warning: No valid bounding boxes after augmentation for Japan_000495.jpg
